# 04 DeepFM 深度学习推荐 + 混合融合

使用 PyTorch 实现 DeepFM（Deep Factorization Machine），
最终与 CF、SVD 进行加权混合得到最终推荐列表，并将结果写回 MySQL。

**混合权重**：CF(0.35) + SVD(0.30) + DeepFM(0.35)

In [1]:
import pandas as pd
import numpy as np
import pickle
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import LabelEncoder

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'使用设备: {DEVICE}')

ratings = pd.read_csv('../data/csv/ratings.csv')
print(ratings.shape)

使用设备: cpu
(18080, 3)


In [2]:
# ── 特征编码 ────────────────────────────────────────────────
le_user = LabelEncoder()
le_item = LabelEncoder()
ratings['user_idx'] = le_user.fit_transform(ratings['user_id'])
ratings['item_idx'] = le_item.fit_transform(ratings['product_id'])
# 标签归一化到 [0,1]
ratings['label']    = (ratings['rating'] - 1) / 4

n_users = ratings['user_idx'].nunique()
n_items = ratings['item_idx'].nunique()
print(f'用户数: {n_users}, 商品数: {n_items}')

用户数: 206, 商品数: 496


In [3]:
# ── PyTorch Dataset ─────────────────────────────────────────
class RatingDataset(Dataset):
    def __init__(self, df):
        self.users  = torch.LongTensor(df['user_idx'].values)
        self.items  = torch.LongTensor(df['item_idx'].values)
        self.labels = torch.FloatTensor(df['label'].values)

    def __len__(self):  return len(self.labels)
    def __getitem__(self, i): return self.users[i], self.items[i], self.labels[i]

from sklearn.model_selection import train_test_split
train_df, val_df = train_test_split(ratings, test_size=0.2, random_state=42)
train_loader = DataLoader(RatingDataset(train_df), batch_size=512, shuffle=True)
val_loader   = DataLoader(RatingDataset(val_df),   batch_size=512)

In [4]:
# ── DeepFM 模型 ─────────────────────────────────────────────
class DeepFM(nn.Module):
    """
    DeepFM = FM 二阶特征交叉  +  DNN 高阶特征
    输入特征: [user_id, item_id] (可扩展为更多 field)
    """
    def __init__(self, n_users, n_items, embed_dim=32, hidden=[128, 64]):
        super().__init__()
        self.user_emb = nn.Embedding(n_users, embed_dim)
        self.item_emb = nn.Embedding(n_items, embed_dim)

        # FM 线性部分
        self.user_bias = nn.Embedding(n_users, 1)
        self.item_bias = nn.Embedding(n_items, 1)
        self.global_bias = nn.Parameter(torch.zeros(1))

        # DNN 部分
        dnn_input = embed_dim * 2
        layers    = []
        for h in hidden:
            layers += [nn.Linear(dnn_input, h), nn.BatchNorm1d(h), nn.ReLU(), nn.Dropout(0.3)]
            dnn_input = h
        layers += [nn.Linear(dnn_input, 1)]
        self.dnn = nn.Sequential(*layers)

        # 输出融合
        self.out = nn.Sigmoid()

    def forward(self, user, item):
        ue = self.user_emb(user)   # (B, D)
        ie = self.item_emb(item)   # (B, D)

        # FM: y = bias + sum of squares trick
        fm_part = (self.user_bias(user) + self.item_bias(item) + self.global_bias).squeeze()
        # FM 二阶: 0.5 * ( (sum v)^2 - sum v^2 )
        sum_sq  = (ue + ie) ** 2
        sq_sum  = ue**2 + ie**2
        fm_part2 = 0.5 * (sum_sq - sq_sum).sum(dim=1)

        # DNN
        dnn_out = self.dnn(torch.cat([ue, ie], dim=1)).squeeze()

        return self.out(fm_part + fm_part2 + dnn_out)

In [5]:
# ── 训练 ─────────────────────────────────────────────────────
model     = DeepFM(n_users, n_items).to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)
criterion = nn.BCELoss()

EPOCHS = 10
for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss = 0
    for users, items, labels in train_loader:
        users, items, labels = users.to(DEVICE), items.to(DEVICE), labels.to(DEVICE)
        preds = model(users, items)
        loss  = criterion(preds, labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    # 验证
    model.eval()
    val_loss = 0
    with torch.no_grad():
        for users, items, labels in val_loader:
            users, items, labels = users.to(DEVICE), items.to(DEVICE), labels.to(DEVICE)
            val_loss += criterion(model(users, items), labels).item()

    print(f'Epoch {epoch:02d}  Train Loss={total_loss/len(train_loader):.4f}  Val Loss={val_loss/len(val_loader):.4f}')

torch.save(model.state_dict(), '../data/csv/deepfm_model.pth')
print('模型已保存')

Epoch 01  Train Loss=2.6013  Val Loss=2.5578
Epoch 02  Train Loss=2.1816  Val Loss=2.1464
Epoch 03  Train Loss=1.8372  Val Loss=1.8410
Epoch 04  Train Loss=1.5487  Val Loss=1.5938
Epoch 05  Train Loss=1.3688  Val Loss=1.4205
Epoch 06  Train Loss=1.2484  Val Loss=1.2860
Epoch 07  Train Loss=1.1647  Val Loss=1.2076
Epoch 08  Train Loss=1.1096  Val Loss=1.1495
Epoch 09  Train Loss=1.0741  Val Loss=1.1023
Epoch 10  Train Loss=1.0476  Val Loss=1.0699
模型已保存


In [6]:
# ── DeepFM 推荐结果生成 ───────────────────────────────────────
model.eval()
all_users_enc = np.arange(n_users)
all_items_enc = np.arange(n_items)

# 用户交互历史（过滤已交互）
user_interacted = ratings.groupby('user_idx')['item_idx'].apply(set).to_dict()

deepfm_results = {}   # {original_user_id: [product_id, ...]}
BATCH = 512

with torch.no_grad():
    for ui in all_users_enc:
        interacted = user_interacted.get(ui, set())
        candidates = [ii for ii in all_items_enc if ii not in interacted]
        if not candidates:
            continue
        scores = []
        for start in range(0, len(candidates), BATCH):
            batch_items = torch.LongTensor(candidates[start:start+BATCH]).to(DEVICE)
            batch_users = torch.LongTensor([ui] * len(batch_items)).to(DEVICE)
            sc = model(batch_users, batch_items).cpu().numpy()
            scores.extend(zip(candidates[start:start+BATCH], sc))
        scores.sort(key=lambda x: x[1], reverse=True)
        orig_uid = le_user.inverse_transform([ui])[0]
        deepfm_results[orig_uid] = [int(le_item.inverse_transform([ii])[0]) for ii, _ in scores[:50]]

with open('../data/csv/deepfm_results.pkl', 'wb') as f:
    pickle.dump(deepfm_results, f)
print(f'DeepFM 推荐结果已保存，覆盖用户数: {len(deepfm_results)}')

DeepFM 推荐结果已保存，覆盖用户数: 206


In [7]:
# ── 混合融合（Hybrid）────────────────────────────────────────
# 加权融合：CF(0.35) + SVD(0.30) + DeepFM(0.35)
CF_W, SVD_W, DEEPFM_W = 0.35, 0.30, 0.35

with open('../data/csv/cf_results.pkl',     'rb') as f: cf_results    = pickle.load(f)
with open('../data/csv/svd_results.pkl',    'rb') as f: svd_results   = pickle.load(f)
with open('../data/csv/deepfm_results.pkl', 'rb') as f: deepfm_res    = pickle.load(f)

def rank_score(items, weight):
    """将排名转为分数：第1位得 weight*1，第N位近似 weight*(1/N)"""
    return {item: weight * (1 / (rank + 1)) for rank, item in enumerate(items)}

hybrid_results = {}
all_users_orig = ratings.user_id.unique()

for uid in all_users_orig:
    scores = {}
    for item, s in rank_score(cf_results.get(uid, [])[:50],    CF_W).items():    scores[item] = scores.get(item,0) + s
    for item, s in rank_score(svd_results.get(uid, [])[:50],   SVD_W).items():   scores[item] = scores.get(item,0) + s
    for item, s in rank_score(deepfm_res.get(uid, [])[:50],    DEEPFM_W).items():scores[item] = scores.get(item,0) + s
    top50 = sorted(scores.items(), key=lambda x: x[1], reverse=True)[:50]
    hybrid_results[uid] = [(item, score) for item, score in top50]

print(f'混合推荐结果计算完成，用户数: {len(hybrid_results)}')

混合推荐结果计算完成，用户数: 206


In [8]:
# ── 将混合推荐结果写回 MySQL ─────────────────────────────────
import os, sys
sys.path.insert(0, os.path.abspath('..'))
import pymysql
from dotenv import load_dotenv
load_dotenv('../backend/.env')

conn = pymysql.connect(
    host=os.getenv('DB_HOST','localhost'), port=int(os.getenv('DB_PORT',3306)),
    user=os.getenv('DB_USER','root'),     password=os.getenv('DB_PASSWORD','root'),
    db=os.getenv('DB_NAME','clothes_recommend'), charset='utf8mb4'
)
cur = conn.cursor()

# ① 先获取 MySQL 里实际存在的 user_id 和 product_id（避免外键报错）
cur.execute('SELECT id FROM users')
valid_users = set(r[0] for r in cur.fetchall())
cur.execute('SELECT id FROM products')
valid_products = set(r[0] for r in cur.fetchall())
print(f'MySQL 有效用户数: {len(valid_users)}, 有效商品数: {len(valid_products)}')

# ② 清空旧的 hybrid 推荐
cur.execute("DELETE FROM recommendations WHERE algo_type='hybrid'")

# ③ 只插入 MySQL 里存在的用户和商品（过滤掉外键不存在的行）
rows = []
for uid, items in hybrid_results.items():
    if int(uid) not in valid_users:
        continue
    for pid, score in items:
        if int(pid) not in valid_products:
            continue
        rows.append((int(uid), int(pid), float(score), 'hybrid'))

cur.executemany("""
    INSERT INTO recommendations (user_id, product_id, score, algo_type)
    VALUES (%s, %s, %s, %s)
""", rows)
conn.commit()
cur.close(); conn.close()

print(f'✅ 写入 recommendations 表: {len(rows)} 条混合推荐结果')
print('   API /api/recommend/personal 现在可以返回个性化推荐了！')


MySQL 有效用户数: 208, 有效商品数: 496
✅ 写入 recommendations 表: 10300 条混合推荐结果
   API /api/recommend/personal 现在可以返回个性化推荐了！


In [9]:
# ── 性能对比总结表 ───────────────────────────────────────────
import pandas as pd
summary = pd.DataFrame([
    {'算法': 'User-CF',      '特点': '基于用户相似度'},
    {'算法': 'Item-CF',      '特点': '基于物品相似度'},
    {'算法': 'SVD',          '特点': '矩阵分解，捕捉隐因子'},
    {'算法': 'SVD++',        '特点': 'SVD 扩展，考虑隐式反馈'},
    {'算法': 'DeepFM',       '特点': 'FM 二阶 + DNN 高阶，端到端学习'},
    {'算法': 'Hybrid Fusion', '特点': 'CF+SVD+DeepFM 加权融合'},
])
print(summary.to_string(index=False))
print('\n🎉 Phase 3 推荐算法开发完成！')

           算法                   特点
      User-CF              基于用户相似度
      Item-CF              基于物品相似度
          SVD           矩阵分解，捕捉隐因子
        SVD++        SVD 扩展，考虑隐式反馈
       DeepFM FM 二阶 + DNN 高阶，端到端学习
Hybrid Fusion   CF+SVD+DeepFM 加权融合

🎉 Phase 3 推荐算法开发完成！
